# 6BUIS017W CWK1 — Market Analysis and Stock Segmentation Using RFM

In [1]:
!git clone https://github.com/aaisha-aamina/crm-sp500-rfm-segmentation
%cd crm-sp500-rfm-segmentation

Cloning into 'crm-sp500-rfm-segmentation'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 20 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 13.59 MiB | 27.44 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/crm-sp500-rfm-segmentation


# Task 1 — Data Preparation and Feature Engineering
## Setup and configuration
`USE_SNAPSHOT = True` makes the notebook read only the saved files in `data/`. Set it to `False` **only** to repeat the one-off download (this would overwrite the snapshot and can change results, because adjusted prices and the constituent list are revised over time).

In [2]:
import io, json, sys, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf

START, END = '2021-01-01', '2025-01-01'     # END is exclusive
BENCH = '^GSPC'
USE_SNAPSHOT = True    # False = download now; True = only read saved files (use this on all later runs)

DATA_DIR = Path('data'); DATA_DIR.mkdir(exist_ok=True)
F_CONST  = DATA_DIR / 'sp500_constituents.csv'
F_HTML   = DATA_DIR / 'sp500_wikipedia_snapshot.html'
F_PRICES = DATA_DIR / 'raw_prices_long.csv.gz'
F_META   = DATA_DIR / 'metadata.json'

versions = {'python': sys.version.split()[0], 'pandas': pd.__version__,
            'numpy': np.__version__, 'yfinance': yf.__version__}
print(versions)

{'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'yfinance': '0.2.66'}


## 1.1 Ticker selection and data acquisition
* **Ticker list:** current S&P 500 constituents read from Wikipedia with `pandas.read_html`; the retrieval date is stored in `metadata.json`. Wikipedia writes share classes as `BRK.B`; Yahoo uses `BRK-B`, so dots are replaced by hyphens.
* **Price source (documented choice):** `yfinance` (Yahoo Finance) for daily Open, High, Low, Close, Adjusted Close and Volume, with `auto_adjust=False` so that raw Close (needed for Monetary) and Adjusted Close (needed for returns) are both kept. The benchmark is `^GSPC`.
* **Period:** 1 Jan 2021 to 1 Jan 2025, end date exclusive.

In [3]:
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

if USE_SNAPSHOT:
    cons = pd.read_csv(F_CONST)
    meta = json.loads(F_META.read_text())
else:
    r = requests.get(WIKI_URL, headers={'User-Agent': 'Mozilla/5.0 (university coursework 6BUIS017W)'}, timeout=30)
    r.raise_for_status()
    F_HTML.write_text(r.text, encoding='utf-8')
    cons = pd.read_html(io.StringIO(r.text))[0]
    # Wikipedia writes BRK.B, Yahoo writes BRK-B
    cons['yf_ticker'] = cons['Symbol'].astype(str).str.strip().str.replace('.', '-', regex=False)
    cons.to_csv(F_CONST, index=False)
    now = datetime.now(timezone.utc)
    meta = {'wikipedia_url': WIKI_URL,
            'retrieval_date_utc': str(now.date()),
            'retrieval_timestamp_utc': now.isoformat(timespec='seconds'),
            'price_source': 'yfinance (auto_adjust=False)',
            'versions': versions, 'start': START, 'end_exclusive': END, 'benchmark': BENCH}
    F_META.write_text(json.dumps(meta, indent=2))

assert not cons['yf_ticker'].duplicated().any()
print(len(cons), 'tickers | retrieved:', meta['retrieval_date_utc'])
cons[['Symbol', 'yf_ticker', 'Security', 'GICS Sector']].head()

503 tickers | retrieved: 2026-11-14


,Symbol,yf_ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology


In [4]:
FIELDS = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

def download_batch(tick_list):
    d = yf.download(tick_list, start=START, end=END, auto_adjust=False, progress=False, threads=True)
    if d is None or d.empty:
        return None
    frames = []
    for f in FIELDS:
        w = d[f].copy(); w.index.name = 'Date'
        frames.append(w.melt(ignore_index=False, var_name='Ticker', value_name=f)
                       .reset_index().set_index(['Date', 'Ticker']))
    return pd.concat(frames, axis=1).reset_index()

if not USE_SNAPSHOT:
    tickers = cons['yf_ticker'].tolist() + [BENCH]
    parts = []
    for i in range(0, len(tickers), 50):
        got = download_batch(tickers[i:i + 50])
        if got is not None:
            parts.append(got)
        print(f'batch {i // 50 + 1}/{-(-len(tickers) // 50)} done')
    long_df = pd.concat(parts, ignore_index=True)

    missing = sorted(set(tickers) - set(long_df['Ticker'].unique()))
    if missing:
        print('retrying:', missing)
        retry = download_batch(missing if len(missing) > 1 else missing + [BENCH])
        if retry is not None:
            long_df = pd.concat([long_df, retry[retry['Ticker'].isin(missing)]], ignore_index=True)

    long_df.to_csv(F_PRICES, index=False, compression='gzip')
    meta['price_download_timestamp_utc'] = datetime.now(timezone.utc).isoformat(timespec='seconds')
    F_META.write_text(json.dumps(meta, indent=2))
    print('saved', F_PRICES, long_df.shape)

**Snapshot check.** From here on the analysis always reads the saved file, never the live download. Shares with no data returned and shares with fewer rows than the benchmark are listed below and carried into the exclusion log in 1.2.

In [6]:
raw = pd.read_csv(F_PRICES, parse_dates=['Date'])
print('rows:', raw.shape[0], '| dates:', raw['Date'].min().date(), '->', raw['Date'].max().date())
print('tickers in file:', raw['Ticker'].nunique(), '| benchmark present:', BENCH in raw['Ticker'].values)
print('file size (MB):', round(F_PRICES.stat().st_size / 1e6, 1))

listed = set(cons['yf_ticker'])
got = set(raw['Ticker'])
failed = sorted(listed - got)
print('listed but NO data returned:', failed)
pd.DataFrame({'ticker': failed, 'reason': 'no data returned by yfinance'}).to_csv(DATA_DIR / 'failed_downloads.csv', index=False)

cov = raw.groupby('Ticker')['Date'].agg(first_date='min', last_date='max', n_obs='count')
cov.to_csv(DATA_DIR / 'download_coverage.csv')
print('shares with fewer rows than the benchmark:', int((cov['n_obs'] < cov.loc[BENCH, 'n_obs']).sum()))
cov.sort_values('n_obs').head()

rows: 506520 | dates: 2021-01-04 -> 2024-12-31
tickers in file: 504 | benchmark present: True
file size (MB): 13.1
listed but NO data returned: []
shares with fewer rows than the benchmark: 0


,first_date,last_date,n_obs
Ticker,,,
A,2021-01-04,2024-12-31,1005
AAPL,2021-01-04,2024-12-31,1005
ABBV,2021-01-04,2024-12-31,1005
ABNB,2021-01-04,2024-12-31,1005
ABT,2021-01-04,2024-12-31,1005
